# Занятие 3, демо 3. Три формы на железе

Три формы одного оператора из занятия 1: по одной позиции, целой матрицей и
блоками. Считают они одно и то же, а стоят по-разному - и порядок зависит от
того, на чём считать.

Здесь два сюжета. Первый: как замер делается правильно и что получается, если
сделать его неправильно. Второй: как меняется порядок форм с ростом длины.

Числа зависят от конкретного железа: на другой карте они будут другими. Смотреть
надо на порядок и на то, как числа растут.

In [ ]:
import torch

torch.set_num_threads(1)

"""Три формы causal linear attention и независимый эталон.

Тот же оператор, что в ДЗ-1. Здесь он нужен как материал демонстрации, а не
как задание, поэтому все три формы даны готовыми.
"""
import itertools

import torch


def reference_by_definition(q, k, v):
    """Определение оператора прямым скалярным суммированием, FP64.

    Матричных произведений и масок здесь нет вообще: это независимая точка
    отсчёта, с которой сравниваются все три формы.
    """
    q, k, v = (t.detach().to(torch.float64) for t in (q, k, v))
    B, H, T, d_k = q.shape
    d_v = v.shape[-1]
    y = torch.zeros(B, H, T, d_v, dtype=torch.float64)

    for b, h, t, p in itertools.product(range(B), range(H), range(T), range(d_v)):
        total = 0.0
        for i in range(t + 1):                      # i <= t: маска включающая
            dot = sum(float(q[b, h, t, a]) * float(k[b, h, i, a])
                      for a in range(d_k))
            total += dot * float(v[b, h, i, p])
        y[b, h, t, p] = total

    return y


def parallel(q, k, v, inclusive=True):
    T = q.shape[-2]
    mask = torch.ones(T, T, dtype=torch.bool, device=q.device)
    mask = mask.tril() if inclusive else mask.tril(diagonal=-1)
    scores = (q @ k.transpose(-1, -2)).masked_fill(~mask, 0.0)
    return scores @ v


def recurrent(q, k, v, inclusive=True):
    T = q.shape[-2]
    state = torch.zeros(*q.shape[:2], v.shape[-1], q.shape[-1],
                        dtype=q.dtype, device=q.device)
    outputs = []

    for t in range(T):
        if inclusive:                                # сначала запись, потом чтение
            state = state + v[..., t, :].unsqueeze(-1) * k[..., t, :].unsqueeze(-2)
            outputs.append((state @ q[..., t, :].unsqueeze(-1)).squeeze(-1))
        else:                                        # сначала чтение, потом запись
            outputs.append((state @ q[..., t, :].unsqueeze(-1)).squeeze(-1))
            state = state + v[..., t, :].unsqueeze(-1) * k[..., t, :].unsqueeze(-2)

    return torch.stack(outputs, dim=-2)


def chunkwise(q, k, v, chunk_size=2, inclusive=True):
    T = q.shape[-2]
    state = torch.zeros(*q.shape[:2], v.shape[-1], q.shape[-1],
                        dtype=q.dtype, device=q.device)
    outputs = []

    for a in range(0, T, chunk_size):
        b = min(a + chunk_size, T)
        qb, kb, vb = q[..., a:b, :], k[..., a:b, :], v[..., a:b, :]
        c = b - a
        mask = torch.ones(c, c, dtype=torch.bool, device=q.device)
        mask = mask.tril() if inclusive else mask.tril(diagonal=-1)
        inner = (qb @ kb.transpose(-1, -2)).masked_fill(~mask, 0.0)
        outputs.append(qb @ state.transpose(-1, -2) + inner @ vb)
        state = state + vb.transpose(-1, -2) @ kb

    return torch.cat(outputs, dim=-2)


def inputs(T=6, d_k=3, d_v=2, B=1, H=1, seed=0):
    g = torch.Generator().manual_seed(seed)
    mk = lambda *s: torch.randn(*s, generator=g, dtype=torch.float64)
    return mk(B, H, T, d_k), mk(B, H, T, d_k), mk(B, H, T, d_v)


def max_diff(a, b) -> float:
    return float((a.to(torch.float64) - b.to(torch.float64)).abs().max())




"""Замер трёх форм линейного оператора на том железе, которое есть.

Если видна карта - замеры идут на ней, через события CUDA и с синхронизацией.
Если карты нет, тот же протокол выполняется на процессоре, только сетка короче:
выводы про порядок величин остаются, абсолютные числа - нет.
"""
import time

import torch


def device():
    """Карта, если она видна, иначе процессор."""
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def describe(dev):
    """Одна строка про то, где считаем."""
    if dev.type == "cuda":
        name = torch.cuda.get_device_name(dev)
        total = torch.cuda.get_device_properties(dev).total_memory / 1024 ** 3
        return f"карта: {name}, памяти {total:.0f} ГиБ, torch {torch.__version__}"
    threads = torch.get_num_threads()
    return f"карты нет, считаем на процессоре в {threads} поток, torch {torch.__version__}"


def grid(dev):
    """Сетка длин и размеров блока: на процессоре короче, чтобы не ждать."""
    if dev.type == "cuda":
        return (512, 1024, 2048, 4096), (16, 64, 256)
    return (256, 512, 1024), (16, 64, 256)


def inputs(T, d_k=64, d_v=64, B=1, H=8, dev=None, seed=0):
    """Одинаковые входы для всех форм: одна партия, 8 голов."""
    dev = dev or device()
    g = torch.Generator().manual_seed(seed)
    mk = lambda *s: torch.randn(*s, generator=g, dtype=torch.float32).to(dev)
    return mk(B, H, T, d_k), mk(B, H, T, d_k), mk(B, H, T, d_v)


def _sync(dev):
    if dev.type == "cuda":
        torch.cuda.synchronize(dev)


def timed(fn, dev, repeats=5, warmup=2):
    """Медиана и разброс в миллисекундах по протоколу: прогрев, синхронизация.

    На карте время берётся событиями CUDA: они ставятся в тот же поток, что и
    работа, и меряют именно её, а не время постановки задачи в очередь.
    """
    for _ in range(warmup):
        fn()
    _sync(dev)

    samples = []
    for _ in range(repeats):
        if dev.type == "cuda":
            start, stop = (torch.cuda.Event(enable_timing=True) for _ in range(2))
            start.record()
            fn()
            stop.record()
            torch.cuda.synchronize(dev)
            samples.append(start.elapsed_time(stop))
        else:
            began = time.perf_counter()
            fn()
            samples.append((time.perf_counter() - began) * 1000)
    samples.sort()
    return samples[len(samples) // 2], samples[-1] - samples[0]


def timed_without_sync(fn, dev, repeats=5, warmup=2):
    """Тот же замер, но без синхронизации - так мерить нельзя.

    На карте вызов возвращает управление сразу после постановки работы в
    очередь. Секундомер снаружи измеряет постановку, а не вычисление.
    """
    for _ in range(warmup):
        fn()
    _sync(dev)

    samples = []
    for _ in range(repeats):
        began = time.perf_counter()
        fn()
        samples.append((time.perf_counter() - began) * 1000)
    samples.sort()
    _sync(dev)
    return samples[len(samples) // 2]


def peak_mib(fn, dev):
    """Пик занятой памяти карты за вызов, в мебибайтах. На процессоре - None."""
    if dev.type != "cuda":
        return None
    torch.cuda.synchronize(dev)
    torch.cuda.reset_peak_memory_stats(dev)
    fn()
    torch.cuda.synchronize(dev)
    return torch.cuda.max_memory_allocated(dev) / 1024 ** 2


def row(name, median, spread, peak):
    memory = "  -" if peak is None else f"{peak:8.1f}"
    return f"  {name:<22} {median:10.3f} {spread:9.3f} {memory}"


def header():
    return (f"  {'форма':<22} {'медиана, мс':>10} {'разброс':>9} "
            f"{'пик, МиБ':>8}\n  " + "-" * 52)

## Где считаем

In [ ]:
dev = device()
print(describe(dev))

lengths, chunks = grid(dev)
print("длины:", lengths)
print("размеры блока:", chunks)

## Как мерить нельзя

На карте вызов возвращает управление, как только работа поставлена в очередь.
Секундомер снаружи в этот момент останавливать нельзя: он измерит постановку
задачи, а не вычисление. Разница бывает в десятки раз.

In [ ]:
T = lengths[-1]
q, k, v = inputs(T, dev=dev)

no_sync = timed_without_sync(lambda: parallel(q, k, v), dev)
honest, spread = timed(lambda: parallel(q, k, v), dev)

print(f"длина {T}")
print(f"  без синхронизации: {no_sync:8.3f} мс")
print(f"  как надо:          {honest:8.3f} мс, разброс {spread:.3f}")
print(f"  отношение:         {honest / no_sync:8.1f}")
if dev.type != "cuda":
    print("  на процессоре синхронизировать нечего, поэтому числа совпадают")

## Протокол

Дальше всё по одному правилу: прогрев, затем несколько повторов, между ними
синхронизация, в таблице медиана и разброс. На карте время берётся событиями
CUDA, на процессоре - обычным секундомером. Память на карте считается как пик
занятого за вызов.

In [ ]:
print(header())
for T in lengths:
    q, k, v = inputs(T, dev=dev)
    for name, call in (("parallel", lambda: parallel(q, k, v)),
                       ("recurrent", lambda: recurrent(q, k, v)),
                       ("chunk 64", lambda: chunkwise(q, k, v, 64))):
        median, spread = timed(call, dev, repeats=3, warmup=1)
        print(row(f"{name}, T={T}", median, spread, peak_mib(call, dev)))
    print()

## Размер блока

Тот же оператор, та же длина, меняется только размер блока.

In [ ]:
T = lengths[len(lengths) // 2]
q, k, v = inputs(T, dev=dev)

print(f"длина {T}")
print(header())
for size in chunks:
    call = lambda size=size: chunkwise(q, k, v, size)
    median, spread = timed(call, dev, repeats=3, warmup=1)
    print(row(f"блок {size}", median, spread, peak_mib(call, dev)))

## Что из этого следует

Асимптотика говорит, у какой формы меньше операций. Она не говорит, какая форма
быстрее на конкретном железе: обход по позициям запускает тысячи мелких задач, и
время уходит на их постановку, а не на арифметику.

Поэтому сравнивать формы имеет смысл только внутри одной реализации и одного
железа, а переносить чужие числа на свою машину нельзя. И поэтому же замер без
прогрева и синхронизации - не замер.